# Early vLLM Smoke Test (Day 3)
**Objective:** Ensure that vLLM can successfully load the FP16/GPTQ version of `LLama-3-8B-Tele-it` on a T4 GPU without OOM errors, and confirm that latency is acceptable (< 4s).
**Bottleneck Mitigation #3**

In [ ]:
!pip install vllm transformers accelerate

In [ ]:
from vllm import LLM, SamplingParams
import time

model_id = "AliMaatouk/LLama-3-8B-Tele-it"
# Note: On Kaggle T4, setting dtype='half' and max_model_len=4096 is crucial to fit in 16GB VRAM
print(f"Loading {model_id} with vLLM...")
start_time = time.time()
llm = LLM(model=model_id, dtype='half', max_model_len=4096, gpu_memory_utilization=0.9)
print(f"Model loaded in {time.time() - start_time:.2f} seconds")

In [ ]:
prompts = [
    "What is RRC connection reconfiguration in 5G?",
    "Explain the difference between SA and NSA architectures.",
    "How does HARQ work in the MAC layer?"
]
sampling_params = SamplingParams(temperature=0.1, max_tokens=256, stop_token_ids=[128001, 128009])

print("Running inference...")
start_time = time.time()
outputs = llm.generate(prompts, sampling_params)
end_time = time.time()

print(f"\nGenerated {len(prompts)} responses in {end_time - start_time:.2f} seconds")
print(f"Average Latency: {(end_time - start_time) / len(prompts):.2f} seconds per query")

for output in outputs:
    print("\n---")
    print(f"Prompt: {output.prompt}")
    print(f"Response: {output.outputs[0].text}")